# Colab starter

Upload `build/birdclef-colab.zip` to `/content` and put competition data on Drive. Prepare `features-train.csv` and `features-valid.csv` with numeric features and binary species targets for the MLP example. The official train.csv is metadata and cannot be used directly.

Set parameters below. Training runs through `!python -m birdclef.models.baseline.fit`; the script owns training, tracking and checkpoint resume.


In [ ]:
import tempfile
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
with tempfile.TemporaryDirectory() as directory:
    wheelhouse = Path(directory)
    with zipfile.ZipFile("/content/birdclef-colab.zip") as archive:
        archive.extractall(wheelhouse)
    wheel = next(wheelhouse.glob("birdclef*.whl"))
    %pip install --find-links "{wheelhouse}" -r "{wheelhouse}/requirements.txt"
    %pip install --force-reinstall --no-deps "{wheel}"


In [ ]:
import os

from birdclef.config import LoaderConfig, RuntimeConfig, TrainConfig, get_config
from birdclef.models import BaselineExperimentConfig, BaselineModelConfig
from birdclef.runtime import apply_environment_defaults, get_secret

apply_environment_defaults()

RUN_NAME = "baseline"
DATASET_ID = "your-data-v1"
WANDB_OFFLINE = False  # True to log locally and sync later with wandb sync.

config = get_config(
    BaselineExperimentConfig,
    model=BaselineModelConfig(feature_columns=["feature"], target_columns=["target"]),
    tracking={"offline": WANDB_OFFLINE},
    runtime=RuntimeConfig(accelerator="auto", devices="auto"),
    loader=LoaderConfig(batch_size=128, val_batch_size=64, num_workers=0),
    train=TrainConfig(max_epochs=5),
)


In [ ]:
import json

# Pass credentials through the environment; saved configs exclude API keys.
if not config.tracking.offline and (api_key := get_secret("WANDB_API_KEY")):
    os.environ["WANDB_API_KEY"] = api_key
elif config.tracking.api_key is not None:
    os.environ["BCLEF_TRACKING__API_KEY"] = config.tracking.api_key.get_secret_value()

config_path = config.data.output_dir / f"{RUN_NAME}-config.json"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(config.safe_dump(), indent=2), encoding="utf-8")
train_path = config.data.competition_dir / "features-train.csv"
valid_path = config.data.competition_dir / "features-valid.csv"


In [ ]:
!python -m birdclef.models.baseline.fit --config "{config_path}" --name "{RUN_NAME}" \
    --dataset-id "{DATASET_ID}" --train "{train_path}" --valid "{valid_path}"
